# Qdrant — inspect & image search

This notebook inspects the local Qdrant DB used by the RAG pipeline and demos figure search by caption embedding.

Open the dashboard at: http://localhost:6333/dashboard

Two collections are used:
- `introduction_textbooks` — text RAG (named vectors: dense `text` + sparse `bm25`)
- `introduction_images` — figure search by caption embedding (named vector: `caption`)

## 0. Setup

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath(".."))
from src.core.config import settings
from src.core.qdrant_store import client
qc = client()
print("Qdrant host:", settings.qdrant_host, settings.qdrant_port)
print("Collections:", [c.name for c in qc.get_collections().collections])

## 1. Text collection — basic stats

In [ ]:
info = qc.get_collection(settings.qdrant_collection_text)
print("Vectors config:", info.config.params.vectors)
print("Sparse vectors:", info.config.params.sparse_vectors)
print("Points:", qc.count(settings.qdrant_collection_text).count)

## 2. Peek payloads — first 5 chunks

In [ ]:
res = qc.scroll(settings.qdrant_collection_text, limit=5, with_payload=True, with_vectors=False)
points, _ = res
for p in points:
    pl = p.payload
    print(f"- {pl.get('book_slug')} / {pl.get('h2_path')} pages={pl.get('page_from')}-{pl.get('page_to')}")
    print(f"  has_formula={pl.get('has_formula')}  text='{pl.get('text','')[:100]}...'")

## 3. Filter — only sections under ch02 with formulas

In [ ]:
from qdrant_client.models import Filter, FieldCondition, MatchValue
flt = Filter(must=[
    FieldCondition(key="book_slug", match=MatchValue(value="islp")),
    FieldCondition(key="chapter_id", match=MatchValue(value="ch02")),
    FieldCondition(key="has_formula", match=MatchValue(value=True)),
])
res, _ = qc.scroll(settings.qdrant_collection_text, scroll_filter=flt, limit=10, with_payload=True)
print(f"Matched: {len(res)} chunks")
for p in res[:5]:
    print(" -", p.payload.get("h2_path"), "n_formulas=", p.payload.get("n_formulas"))

## 4. Hybrid query — dense + sparse via Qdrant RRF

In [ ]:
from src.services.retrieval.retrievers import QdrantHybridChildRetriever
child = QdrantHybridChildRetriever(book_slug="islp", k=10)
docs = child.invoke("bias variance tradeoff")
for d in docs[:5]:
    print(d.metadata.get("h2_path"), "-", d.page_content[:120])

## 5. Image collection — basic stats + scroll

In [ ]:
print("Images:", qc.count(settings.qdrant_collection_images).count)
res, _ = qc.scroll(settings.qdrant_collection_images, limit=5, with_payload=True)
for p in res:
    pl = p.payload
    print(f"- {pl['image_name']}  page={pl.get('page')}")
    print(f"  caption: {pl.get('image_reference','')[:120]}")
    print(f"  path: {pl.get('image_path')}")

## 6. Image semantic search by caption

In [ ]:
from src.services.retrieval.retrievers import search_images
results = search_images("boxplot of wage vs education", book_slug="islp", k=5)
for r in results:
    print(f"score={r['score']:.3f}  {r['image_name']}  page={r.get('page')}")
    print(f"  ref: {r['image_reference'][:120]}")

## 7. Preview an image inline

In [ ]:
from pathlib import Path
from IPython.display import Image, display
top = results[0]
img_path = top["image_path"]
if Path(img_path).exists():
    display(Image(filename=img_path, width=500))
else:
    print("File not on disk:", img_path)

## 8. Dashboard tip

Open http://localhost:6333/dashboard in your browser:

1. Pick a collection (`introduction_textbooks` or `introduction_images`).
2. Browse points visually — inspect payloads, vectors, and IDs.
3. Use the **Filter** panel in the UI to run the same `book_slug` / `chapter_id` / `has_formula` filters interactively.
4. Run ad-hoc similarity queries from the dashboard to debug retrieval without leaving the browser.